In [1]:
import pandas as pd
from pathlib import Path

# Folder containing your Parquet files
prepared_dir = Path("data/prepared")
parquet_files = list(prepared_dir.glob("*_with_features.parquet"))

results = {}

for file in parquet_files:
    ticker = file.stem.replace("_with_features", "")
    df = pd.read_parquet(file)

    if "signal_12m_return" not in df.columns or "Close" not in df.columns:
        print(f"Skipping {ticker}: missing required columns")
        continue

    # Calculate returns
    df["daily_return"] = df["Close"].pct_change()
    df["strategy_return"] = df["daily_return"].shift(-1) * df["signal_12m_return"]
    df["cumulative_strategy_return"] = (1 + df["strategy_return"].fillna(0)).cumprod()
    df["cumulative_market_return"] = (1 + df["daily_return"].fillna(0)).cumprod()

    if df["strategy_return"].std() == 0:
        print(f"Skipping {ticker}: strategy return std is 0")
        continue

    # Performance metrics
    total_return = df["cumulative_strategy_return"].iloc[-1] - 1
    volatility = df["strategy_return"].std() * (252 ** 0.5)
    sharpe = df["strategy_return"].mean() / df["strategy_return"].std() * (252 ** 0.5)

    results[ticker] = {
        "Total Return": total_return,
        "Volatility": volatility,
        "Sharpe Ratio": sharpe
    }

# Final DataFrame
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by="Sharpe Ratio", ascending=False)
results_df


,Total Return,Volatility,Sharpe Ratio
EURUSD=X,0.051658,0.072290,0.369198
QQQ,0.078763,0.212467,0.287790
GLD,-0.027203,0.158667,-0.008858
SPY,-0.036352,0.165909,-0.029207
^VIX,-0.910569,1.397561,-0.235980
AGG,-0.039100,0.060442,-0.303740
XLF,-0.130245,0.168493,-0.335083
IWM,-0.195096,0.220083,-0.388774
XLV,-0.144919,0.135139,-0.518843
MSFT,-0.256248,0.230347,-0.534031
